In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
pio.templates.default = "plotly_white"
%pip install -U nbformat

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
online_retail = fetch_ucirepo(id=352) 
  
# data (as pandas dataframes) 
X = online_retail.data.features 
y = online_retail.data.targets 
  
# metadata 
print(online_retail.metadata) 
  
# variable information 
print(online_retail.variables) 


{'uci_id': 352, 'name': 'Online Retail', 'repository_url': 'https://archive.ics.uci.edu/dataset/352/online+retail', 'data_url': 'https://archive.ics.uci.edu/static/public/352/data.csv', 'abstract': 'This is a transactional data set which contains all the transactions occurring between 01/12/2010 and 09/12/2011 for a UK-based and registered non-store online retail.', 'area': 'Business', 'tasks': ['Classification', 'Clustering'], 'characteristics': ['Multivariate', 'Sequential', 'Time-Series'], 'num_instances': 541909, 'num_features': 6, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': None, 'index_col': ['InvoiceNo', 'StockCode'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2015, 'last_updated': 'Mon Oct 21 2024', 'dataset_doi': '10.24432/C5BW33', 'creators': ['Daqing Chen'], 'intro_paper': {'ID': 361, 'type': 'NATIVE', 'title': 'Data mining for the online retail industry: A case study of RFM model-based customer segmenta

In [ ]:
data = online_retail.data.features

print(data.head())

                           Description  Quantity     InvoiceDate  UnitPrice  \
0   WHITE HANGING HEART T-LIGHT HOLDER         6  12/1/2010 8:26       2.55   
1                  WHITE METAL LANTERN         6  12/1/2010 8:26       3.39   
2       CREAM CUPID HEARTS COAT HANGER         8  12/1/2010 8:26       2.75   
3  KNITTED UNION FLAG HOT WATER BOTTLE         6  12/1/2010 8:26       3.39   
4       RED WOOLLY HOTTIE WHITE HEART.         6  12/1/2010 8:26       3.39   

   CustomerID         Country  
0     17850.0  United Kingdom  
1     17850.0  United Kingdom  
2     17850.0  United Kingdom  
3     17850.0  United Kingdom  
4     17850.0  United Kingdom  


In [ ]:
# Convert InvoiceDate to datetime
data['InvoiceDate'] = pd.to_datetime(data['InvoiceDate'])

# Display basic information
print(data.info())
print(data.head())

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Description  540455 non-null  str           
 1   Quantity     541909 non-null  int64         
 2   InvoiceDate  541909 non-null  datetime64[us]
 3   UnitPrice    541909 non-null  float64       
 4   CustomerID   406829 non-null  float64       
 5   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(2)
memory usage: 24.8 MB
None
                           Description  Quantity         InvoiceDate  \
0   WHITE HANGING HEART T-LIGHT HOLDER         6 2010-12-01 08:26:00   
1                  WHITE METAL LANTERN         6 2010-12-01 08:26:00   
2       CREAM CUPID HEARTS COAT HANGER         8 2010-12-01 08:26:00   
3  KNITTED UNION FLAG HOT WATER BOTTLE         6 2010-12-01 08:26:00   
4       RED WOOLLY HOTTIE WHITE HEART.         6 2010-12-0

In [ ]:
ids = online_retail.data.ids

print(ids.columns)
print(ids.head())

Index(['InvoiceNo', 'StockCode'], dtype='str')
  InvoiceNo StockCode
0    536365    85123A
1    536365     71053
2    536365    84406B
3    536365    84029G
4    536365    84029E


In [ ]:
import pandas as pd

data = pd.concat(
    [online_retail.data.ids, online_retail.data.features],
    axis=1
)

print(data.head())
print(data.columns)

  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='str')


In [ ]:
print(data.columns.tolist())

['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


In [ ]:
# Remove rows without CustomerID
data = data.dropna(subset=['CustomerID']).copy()

# Remove cancelled/returned transactions
data = data[~data['InvoiceNo'].astype(str).str.startswith('C')].copy()

# Remove zero or negative quantities
data = data[data['Quantity'] > 0].copy()

# Remove zero or negative prices
data = data[data['UnitPrice'] > 0].copy()

# Calculate MonetaryValue
data['MonetaryValue'] = data['Quantity'] * data['UnitPrice']

# Reset index
data.reset_index(drop=True, inplace=True)

print(data.info())

<class 'pandas.DataFrame'>
RangeIndex: 397884 entries, 0 to 397883
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   InvoiceNo      397884 non-null  str           
 1   StockCode      397884 non-null  str           
 2   Description    397884 non-null  str           
 3   Quantity       397884 non-null  int64         
 4   InvoiceDate    397884 non-null  datetime64[us]
 5   UnitPrice      397884 non-null  float64       
 6   CustomerID     397884 non-null  float64       
 7   Country        397884 non-null  str           
 8   MonetaryValue  397884 non-null  float64       
dtypes: datetime64[us](1), float64(3), int64(1), str(4)
memory usage: 27.3 MB
None


In [ ]:
#Reference_date = latest transaction date
reference_date = pd.Timestamp(data['InvoiceDate'].max())

#Calculate Customer-level RFM
rfm = data.groupby('CustomerID').agg(
    Recency=('InvoiceDate' , lambda x: (reference_date - pd.Timestamp(x.max())).days),
    Frequency=('InvoiceNo', 'nunique'),
    MonetaryValue=('MonetaryValue', 'sum')
).reset_index()

print(rfm.head())

   CustomerID  Recency  Frequency  MonetaryValue
0     12346.0      325          1       77183.60
1     12347.0        1          7        4310.00
2     12348.0       74          4        1797.24
3     12349.0       18          1        1757.55
4     12350.0      309          1         334.40


In [ ]:
# Define scoring criteria
recency_scores = [5, 4, 3, 2, 1]
frequency_scores = [1, 2, 3, 4, 5]
monetary_scores = [1, 2, 3, 4, 5]

# Calculate Recency score
rfm['RecencyScore'] = pd.qcut(
    rfm['Recency'],
    q=5,
    labels=recency_scores
)

# Calculate Frequency score
rfm['FrequencyScore'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    q=5,
    labels=frequency_scores
)

# Calculate Monetary score
rfm['MonetaryScore'] = pd.qcut(
    rfm['MonetaryValue'].rank(method='first'),
    q=5,
    labels=monetary_scores
)

# Convert scores to integers
rfm['RecencyScore'] = rfm['RecencyScore'].astype(int)
rfm['FrequencyScore'] = rfm['FrequencyScore'].astype(int)
rfm['MonetaryScore'] = rfm['MonetaryScore'].astype(int)

# Calculate total RFM score
rfm['RFM_Score'] = (
    rfm['RecencyScore']
    + rfm['FrequencyScore']
    + rfm['MonetaryScore']
)

# Create Value Segment
segment_labels = ['Low-Value', 'Mid-Value', 'High-Value']

rfm['Value Segment'] = pd.qcut(
    rfm['RFM_Score'],
    q=3,
    labels=segment_labels
)

print(rfm.head())

   CustomerID  Recency  Frequency  MonetaryValue  RecencyScore  \
0     12346.0      325          1       77183.60             1   
1     12347.0        1          7        4310.00             5   
2     12348.0       74          4        1797.24             2   
3     12349.0       18          1        1757.55             4   
4     12350.0      309          1         334.40             1   

   FrequencyScore  MonetaryScore  RFM_Score Value Segment  
0               1              5          7     Low-Value  
1               5              5         15    High-Value  
2               4              4         10     Mid-Value  
3               1              4          9     Mid-Value  
4               1              2          4     Low-Value  


In [ ]:
import nbformat
print(nbformat.__version__)

5.11.1


In [ ]:
# Count customers in each value segment
segment_counts = (
    rfm['Value Segment']
    .value_counts()
    .reset_index()
)

segment_counts.columns = ['Value Segment', 'Count']

#Create bar chart
fig_segment_dist = px.bar(
    segment_counts,
    x='Value Segment',
    y='Count',
    color='Value Segment',
    color_discrete_sequence=px.colors.qualitative.Pastel,
    title='RFM Segment Distribution'
)

fig_segment_dist.update_layout(
    xaxis_title='RFM Value Segment',
    yaxis_title='Count',
    showlegend=False
)

fig_segment_dist.show()

In [ ]:
#Create RFM Customer Segments
rfm['RFM Customer Segment'] = ''

#Champions
rfm.loc[
    rfm['RFM_Score'] >= 9,
    'RFM Customer Segments'
] = 'Champions'

# Potential Loyalists
rfm.loc[
    (rfm['RFM_Score'] >= 6) &
    (rfm['RFM_Score'] < 9),
    'RFM Customer Segments'
] = 'Potential Loyalists'

# At Risk Customers
rfm.loc[
    (rfm['RFM_Score'] >= 5) &
    (rfm['RFM_Score'] < 6),
    'RFM Customer Segments'
] = 'At Risk Customers'

# Can't Lose
rfm.loc[
    (rfm['RFM_Score'] >= 4) &
    (rfm['RFM_Score'] < 5),
    'RFM Customer Segments'
] = "Can't Lose"

# Lost
rfm.loc[
    (rfm['RFM_Score'] >= 3) &
    (rfm['RFM_Score'] < 4),
    'RFM Customer Segments'
] = 'Lost'

#Display results
print(rfm[
        ['CustomerID', 'RFM_Score', 'RFM Customer Segments']
    ]
)

      CustomerID  RFM_Score RFM Customer Segments
0        12346.0          7   Potential Loyalists
1        12347.0         15             Champions
2        12348.0         10             Champions
3        12349.0          9             Champions
4        12350.0          4            Can't Lose
...          ...        ...                   ...
4333     18280.0          4            Can't Lose
4334     18281.0          4            Can't Lose
4335     18282.0          9             Champions
4336     18283.0         15             Champions
4337     18287.0         11             Champions

[4338 rows x 3 columns]


In [ ]:
print(rfm['RFM Customer Segments'].value_counts())

RFM Customer Segments
Champions              2277
Potential Loyalists    1178
Can't Lose              362
At Risk Customers       338
Lost                    183
Name: count, dtype: int64


In [ ]:
# Count customers by Value Segment and RFM Customer Segment
segment_product_counts = (
    rfm.groupby(
        ['Value Segment', 'RFM Customer Segments']
    )
    .size()
    .reset_index(name='Count')
)

# Sort by count
segment_product_counts = segment_product_counts.sort_values(
    'Count',
    ascending=False
)

# Create treemap
fig_treemap_segment_product = px.treemap(
    segment_product_counts,
    path=['Value Segment', 'RFM Customer Segments'],
    values='Count',
    color='Value Segment',
    color_discrete_sequence=px.colors.qualitative.Pastel,
    title='RFM Customer Segments by Value'
)

fig_treemap_segment_product.show()

In [ ]:
# Filter only Champions
champions_segment = rfm[
    rfm['RFM Customer Segments'] == 'Champions'
]

# Create box plots
fig = go.Figure()

fig.add_trace(
    go.Box(
        y=champions_segment['RecencyScore'],
        name='Recency'
    )
)

fig.add_trace(
    go.Box(
        y=champions_segment['FrequencyScore'],
        name='Frequency'
    )
)

fig.add_trace(
    go.Box(
        y=champions_segment['MonetaryScore'],
        name='Monetary'
    )
)

fig.update_layout(
    title='Distribution of RFM Values within Champions Segment',
    yaxis_title='RFM Value',
    showlegend=True
)

fig.show()

In [ ]:
#Calculate correlation matrix
correlation_matrix = champions_segment[
    ['RecencyScore', 'FrequencyScore', 'MonetaryScore']
].corr()

print(correlation_matrix)

                RecencyScore  FrequencyScore  MonetaryScore
RecencyScore         1.00000        0.138380       0.012940
FrequencyScore       0.13838        1.000000       0.533737
MonetaryScore        0.01294        0.533737       1.000000


In [ ]:
print(champions_segment.shape)

(2277, 11)


In [ ]:
print(champions_segment[
    ['RecencyScore', 'FrequencyScore', 'MonetaryScore']
].nunique())

RecencyScore      5
FrequencyScore    5
MonetaryScore     5
dtype: int64


In [ ]:
#Count customers in each RFM Customer Segments
customer_segment_counts = (
    rfm['RFM Customer Segments'].value_counts()
)

print(customer_segment_counts)

RFM Customer Segments
Champions              2277
Potential Loyalists    1178
Can't Lose              362
At Risk Customers       338
Lost                    183
Name: count, dtype: int64


In [ ]:
pastel_colors = px.colors.qualitative.Pastel

fig = go.Figure(
    data=[
        go.Bar(
            x=customer_segment_counts.index,
            y=customer_segment_counts.values,
            marker=dict(color=pastel_colors)
        )
    ]
)

# Set the color of the Champions segment as a different color
champions_color = 'rgb(158, 202, 225)'
fig.update_traces(marker_color=[champions_color if segment == 'Champions' else pastel_colors[i]
                                for i, segment in enumerate(segment_counts.index)],
                  marker_line_color='rgb(8, 48, 107)',
                  marker_line_width=1.5, opacity=0.6)

fig.update_layout(
    title='Comparison of RFM Customer Segments',
    xaxis_title='RFM Customer Segments',
    yaxis_title='Number of Customers',
    showlegend=False
)

fig.show()

In [ ]:
# Calculate the average Recency, Frequency, and Monetary scores for each segment
segment_scores = (
    rfm.groupby('RFM Customer Segments')
    [['RecencyScore', 'FrequencyScore', 'MonetaryScore']]
    .mean().reset_index()
)

# Create a grouped bar chart to compare segment scores
fig = go.Figure()

# Recency
fig.add_trace(go.Bar(
    x=segment_scores['RFM Customer Segments'],
    y=segment_scores['RecencyScore'],
    name='Recency Score'
))

# Frequency
fig.add_trace(go.Bar(
    x=segment_scores['RFM Customer Segments'],
    y=segment_scores['FrequencyScore'],
    name='Frequency Score'
))

# Monetary
fig.add_trace(go.Bar(
    x=segment_scores['RFM Customer Segments'],
    y=segment_scores['MonetaryScore'],
    name='Monetary Score'
))

# Layout
fig.update_layout(
    title='Comparison of RFM Segments based on Recency, Frequency, and Monetary Scores',
    xaxis_title='RFM Segments',
    yaxis_title='Average RFM Score',
    barmode='group',
    showlegend=True
)

fig.show()